In [1]:
import pypsa
import fbmc  # noqa: F401  not unused; registers pypsa.Network.fbmc accessor
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import networkx as nx


In [5]:
nodal_net = pypsa.Network("base_s_25_elec_Ep100.nc")
nodal_net.buses.rename(columns={"country": "zone_name"}, inplace=True)
nodal_net.buses.loc["DK1 0", "zone_name"] = "DK-10"
nodal_net.buses.loc["DK1 3", "zone_name"] = "DK-13"
nodal_net.buses.loc["DK3 0", "zone_name"] = "DK-30"
nodal_net.buses.loc["GB2 0", "zone_name"] = "NIR"
nodal_net.remove("Bus", "DK1 3")
bus_zone_map = nodal_net.buses['zone_name']

INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, lines, links, loads, storage_units


In [8]:
nodal_net.buses.zone_name.unique()

array(['BE', 'DE', 'DK-10', 'DK-30', 'FR', 'GB', 'NIR', 'IE', 'LU', 'NL'],
      dtype=object)

In [20]:
country_iteration = "BE"

In [21]:
zone_iteration = bus_zone_map[bus_zone_map == "BE"].index[0]

In [23]:
zone_iteration

'BE1 0'

In [26]:
nodal_net.generators[nodal_net.generators.carrier.isin(["solar", 'onwind'])].query('bus == @zone_iteration').p_nom

Generator
BE1 0 0 solar      937.60
BE1 0 0 onwind    1753.56
Name: p_nom, dtype: float64

In [5]:
nodal_net.generators.carrier.unique()

array(['CCGT', 'biomass', 'nuclear', 'oil', 'waste', 'OCGT', 'coal',
       'geothermal', 'lignite', 'offwind-dc', 'solar', 'offwind-ac',
       'offwind-float', 'solar-hsat', 'onwind'], dtype=object)

In [5]:
def freeze_expansion(network):
    """
    Replace nominal capacities by their optimized values and
    disable further expansion.

    Parameters
    ----------
    network : pypsa.Network
    """

    for comp in network.components.values():
        df = getattr(network, comp.list_name, None)

        if df is None or df.empty:
            continue

        cols = df.columns

        # p_nom -> p_nom_opt
        if "p_nom" in cols and "p_nom_opt" in cols:
            df["p_nom"] = df["p_nom_opt"]
            if "p_nom_extendable" in cols:
                df["p_nom_extendable"] = False

        # s_nom -> s_nom_opt
        if "s_nom" in cols and "s_nom_opt" in cols:
            df["s_nom"] = df["s_nom_opt"]
            if "s_nom_extendable" in cols:
                df["s_nom_extendable"] = False

        # e_nom -> e_nom_opt
        if "e_nom" in cols and "e_nom_opt" in cols:
            df["e_nom"] = df["e_nom_opt"]
            if "e_nom_extendable" in cols:
                df["e_nom_extendable"] = False

    return network

In [6]:
freeze_expansion(nodal_net)

PyPSA Network 'Unnamed Network'
-------------------------------
Components:
 - Bus: 25
 - Carrier: 19
 - Generator: 256
 - Line: 46
 - Link: 36
 - Load: 25
 - StorageUnit: 25
Snapshots: 8736

In [8]:

zonal_net = nodal_net.fbmc.to_zonal(bus_zone_map)

In [63]:
zonal_net.set_snapshots(zonal_net.snapshots[:2000])

In [9]:
config = fbmc.FBMCConfig(
    base_case_strategy=fbmc.BaseCaseStrategy.ZERO_FLOWS,
    add_security_constraints=False,
    gsk_strategy=fbmc.GSKStrategy.P_NOM,
    security_constraint_bodf_size_threshold=0.1,
    min_ram=0.1,
    reliability_margin_factor=0.0,
    advanced_hybrid_coupling_flag= True,
    #solver_kwargs={'solver_name': 'gurobi', 'OutputFlag': 0},
    solver_kwargs=solver_options
)

zonal_net.fbmc.create_model(nodal_net, config=config)

       'relation/15772117-320-DC', 'relation/15781671-525-DC',
       'relation/2127794-270-DC', 'relation/2505320-400-DC',
       'relation/5487095-400-DC', 'relation/6914309-500-DC',
       'relation/8184641-200-DC', 'relation/8185420-320-DC',
       'relation/8185487-400-DC', 'relation/8193755-320-DC', 'TYNDP2020_17',
       'TYNDP2020_32', 'TYNDP2020_36', 'DC2', 'TYNDP2024_153', 'TYNDP2024_285',
       'TYNDP2022_286'],
      dtype='object', name='Link')
INFO:root:Determined 4 sub-networks in the base case nodal network.
INFO:root:Created optimization model without meshed split.


Fixed load has coordinate Bus


Linopy LP model

Variables:
----------
 * Generator-p (snapshot, Generator)
 * Link-p (snapshot, Link)
 * StorageUnit-p_dispatch (snapshot, StorageUnit)
 * StorageUnit-p_store (snapshot, StorageUnit)
 * StorageUnit-state_of_charge (snapshot, StorageUnit)
 * Zone-p (snapshot, Zone)

Constraints:
------------
 * Generator-fix-p-lower (snapshot, Generator-fix)
 * Generator-fix-p-upper (snapshot, Generator-fix)
 * Link-fix-p-lower (snapshot, Link-fix)
 * Link-fix-p-upper (snapshot, Link-fix)
 * StorageUnit-fix-p_dispatch-lower (snapshot, StorageUnit-fix)
 * StorageUnit-fix-p_dispatch-upper (snapshot, StorageUnit-fix)
 * StorageUnit-fix-p_store-lower (snapshot, StorageUnit-fix)
 * StorageUnit-fix-p_store-upper (snapshot, StorageUnit-fix)
 * StorageUnit-fix-state_of_charge-lower (snapshot, StorageUnit-fix)
 * StorageUnit-fix-state_of_charge-upper (snapshot, StorageUnit-fix)
 * StorageUnit-energy_balance (snapshot, StorageUnit)
 * Zone-definition (snapshot, Zone)
 * CNEC-upper-RAM-subnet-0 (s

In [10]:
zonal_net.model.solve(**config.solver_kwargs)
result = zonal_net.fbmc.results()
print(result)

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - threads: 32
 - method: 2
 - crossover: 0
 - BarConvTol: 1e-05
 - Seed: 123
 - AggFill: 0
 - PreDual: 0
 - GURO_PAR_BARDENSETHRESH: 200
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 6/6 [00:00<00:00, 94.29it/s]
INFO:linopy.io: Writing time: 1.2s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2838876


INFO:gurobipy:Set parameter LicenseID to value 2838876


...

Optimal objective 2.36566571e+10


INFO:gurobipy:Optimal objective 2.36566571e+10


INFO:gurobipy:
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 3144960 primals, 7207200 duals
Objective: 2.37e+10
Solver model: available
Solver message: 2



FBMCResult
  zonal_net: Unnamed Network | snapshots=8736 | components=[Bus:10, Generator:256, Load:25, Link:19, StorageUnit:25]
  base_case: Unnamed Network | snapshots=8736 | components=[Bus:25, Generator:256, Load:25, Line:46, Link:36, StorageUnit:25, SubNetwork:4]
  net_positions: DataFrame(8736, 10)
  dispatch_results: DispatchResult object with attrs: 
  generators_p: (8736, 256) snapshots x generators, 
  storage_units_p: (8736, 25) snapshots x storage units, 
  links_p0: (8736, 19) snapshots x links, 
  storage_levels: (8736, 25) snapshots x storage levels, 
  water_values: (8736, 25) snapshots x water values
  fbmc_parameters: 2 subnet(s) [0, 2]


/Users/ng-work/.pyenv/versions/pypsa-fbmc/lib/python3.11/site-packages/linopy/common.py:492: UserWarning: Coordinates across variables not equal. Perform outer join.
  warn(
/Users/ng-work/.pyenv/versions/pypsa-fbmc/lib/python3.11/site-packages/linopy/common.py:492: UserWarning: Coordinates across variables not equal. Perform outer join.
  warn(


In [12]:
from fbmc.post_processing.main import process_results

In [13]:
process_results(result, rd_cost= None, rd_dispatch= None, save_path = "/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt", config = config)

INFO:pypsa.network.io:Exported network 'Unnamed Network'saved to '/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/fbmc_network.nc contains: links, storage_units, generators, buses, loads, carriers


{'zonal_market_prices': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/zonal_market_prices.csv'),
 'load_shedding_zone_p': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/load_shedding_zone_p.csv'),
 'generation_mix': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/generation_mix.csv'),
 'storage_mix': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/storage_mix.csv'),
 'storage_units_p': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/storage_units_p.csv'),
 'summary': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/summary.json'),
 'linopy_model': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/linopy_model.nc'),
 'net_positions_zone_p': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/net_positions_zo

In [14]:
result.zonal_net.model.constraints["Bus-nodal_balance"].dual.to_pandas()

KeyError: 'Bus-nodal_balance'

In [34]:
result.zonal_net.model.constraints["Zone-definition"].loc[:,"IE"].dual.to_pandas()

snapshot
2013-01-01 00:00:00    0.0
2013-01-01 01:00:00    0.0
2013-01-01 02:00:00    0.0
2013-01-01 03:00:00    0.0
2013-01-01 04:00:00    0.0
                      ... 
2013-12-30 19:00:00    0.0
2013-12-30 20:00:00    0.0
2013-12-30 21:00:00    0.0
2013-12-30 22:00:00    0.0
2013-12-30 23:00:00    0.0
Length: 8736, dtype: float64

In [21]:
result.zonal_net.statistics.expanded_capacity()

component    carrier                 
Generator    Combined-Cycle Gas          0.0
             Offshore Wind (AC)          0.0
             Offshore Wind (DC)          0.0
             Offshore Wind (Floating)    0.0
             Onshore Wind                0.0
             Open-Cycle Gas              0.0
             Solar                       0.0
             biomass                     0.0
             coal                        0.0
             geothermal                  0.0
             lignite                     0.0
             nuclear                     0.0
             oil                         0.0
             solar-hsat                  0.0
             waste                       0.0
Link         DC                          0.0
StorageUnit  Battery Storage             0.0
dtype: float64

In [128]:
result.zonal_net.model.constraints["Generator-ext-p-upper"]

Constraint `Generator-ext-p-upper` [snapshot: 8736, Generator-ext: 172]:
------------------------------------------------------------------------
[2013-01-01 00:00:00, BE1 0 CCGT]: +1 Generator-p[2013-01-01 00:00:00, BE1 0 CCGT] - 1 Generator-p_nom[BE1 0 CCGT]                  ≤ -0.0
[2013-01-01 00:00:00, BE1 0 nuclear]: +1 Generator-p[2013-01-01 00:00:00, BE1 0 nuclear] - 0.781 Generator-p_nom[BE1 0 nuclear]     ≤ -0.0
[2013-01-01 00:00:00, DE1 0 CCGT]: +1 Generator-p[2013-01-01 00:00:00, DE1 0 CCGT] - 1 Generator-p_nom[DE1 0 CCGT]                  ≤ -0.0
[2013-01-01 00:00:00, DE1 0 OCGT]: +1 Generator-p[2013-01-01 00:00:00, DE1 0 OCGT] - 1 Generator-p_nom[DE1 0 OCGT]                  ≤ -0.0
[2013-01-01 00:00:00, DE1 1 CCGT]: +1 Generator-p[2013-01-01 00:00:00, DE1 1 CCGT] - 1 Generator-p_nom[DE1 1 CCGT]                  ≤ -0.0
[2013-01-01 00:00:00, DE1 1 OCGT]: +1 Generator-p[2013-01-01 00:00:00, DE1 1 OCGT] - 1 Generator-p_nom[DE1 1 OCGT]                  ≤ -0.0
[2013-01-01 00:00:00

In [117]:
nodal_net.statistics.expanded_capacity()

component    carrier                 
Generator    Combined-Cycle Gas               0.89296
             Offshore Wind (AC)               1.34084
             Offshore Wind (DC)               4.58522
             Offshore Wind (Floating)         3.49939
             Onshore Wind                 38837.28688
             Open-Cycle Gas                   0.10507
             Solar                        19085.71321
             biomass                          0.00000
             coal                             0.00000
             geothermal                       0.00000
             lignite                          0.00000
             nuclear                          0.42779
             oil                              0.00000
             solar-hsat                  135482.99670
             waste                            0.00000
Line         AC                           85080.20262
Link         DC                              14.58120
StorageUnit  Battery Storage              26

In [118]:
result.zonal_net.statistics.expanded_capacity()

component    carrier                 
Generator    Combined-Cycle Gas               0.89296
             Offshore Wind (AC)               1.34084
             Offshore Wind (DC)               4.58522
             Offshore Wind (Floating)         3.49939
             Onshore Wind                 38837.28688
             Open-Cycle Gas                   0.10507
             Solar                        19085.71321
             biomass                          0.00000
             coal                             0.00000
             geothermal                       0.00000
             lignite                          0.00000
             nuclear                          0.42779
             oil                              0.00000
             solar-hsat                  135482.99670
             waste                            0.00000
Link         DC                               3.68863
StorageUnit  Battery Storage              26355.85776
dtype: float64